# LOGOS v3 — Production GPU Training
## 100MB Real Data | Full Benchmark | Physics Evaluation

| Config | Value | Notes |
|--------|-------|-------|
| **Model** | d=256, L=6, H=8 | ~60M params |
| **Seq len** | 256 | longer context |
| **Dataset** | 100MB TinyStories | real text |
| **Optimizer** | Hybrid SHM v9 | Hamiltonian + Langevin |
| **Loss** | Free Energy F=CE-T·S | thermodynamic |
| **GEMM** | Vedic (Gunitasamuchayah) | verified |
| **Embeddings** | Hyperbolic Poincaré Ball | Phase 2 |
| **KV Cache** | Nikhilam INT8 4x | Phase 2 |
| **Inference** | Feynman Beam Search | Phase 3 |

**What this measures:**
- Speed: tok/s, ms/step, M-tok/min
- VRAM: peak usage, INT8 savings, GPU utilization %
- Convergence: F curve, PPL, CE vs Entropy decomposition
- Physics: α_H/α_L annealing, FDT verification, gradient health
- Vedic: Gunitasamuchayah pass rate over training
- Generation: Feynman beam search at ħ=0.3/1.0/1.5

In [ ]:
# CELL 1 — Environment & GPU Detection
import os, subprocess, re, math, time, glob, json, threading, shutil
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

WORK_DIR   = '/kaggle/working/LOGOS'
OUT_DIR    = '/kaggle/working/logos_v3_output'
TRAIN_FILE = '/kaggle/working/dataset_100mb.txt'
LOG_FILE   = '/kaggle/working/train_log_v3.txt'
BUILD_LOG  = '/kaggle/working/build_log.txt'
os.makedirs(OUT_DIR, exist_ok=True)

print('=== LOGOS v3 Production GPU Training ===')
os.system('nvidia-smi --query-gpu=name,memory.total,memory.free,compute_cap --format=csv,noheader')

r = subprocess.run(['nvidia-smi','--query-gpu=memory.total,name,compute_cap',
                    '--format=csv,noheader,nounits'], capture_output=True, text=True)
parts  = r.stdout.strip().split('\n')[0].split(',')
VRAM_MB  = int(parts[0].strip())
GPU_NAME = parts[1].strip()
GPU_CC   = parts[2].strip().replace('.', '')

# Auto config by VRAM
if VRAM_MB >= 8000:
    D_MODEL, N_LAYERS, N_HEADS, SEQ_LEN = 256, 6, 8, 256
    CONFIG_LABEL = 'FULL (d=256 L=6 H=8 seq=256)'
elif VRAM_MB >= 4000:
    D_MODEL, N_LAYERS, N_HEADS, SEQ_LEN = 128, 4, 8, 128
    CONFIG_LABEL = 'MID  (d=128 L=4 H=8 seq=128)'
else:
    D_MODEL, N_LAYERS, N_HEADS, SEQ_LEN = 64, 2, 4, 64
    CONFIG_LABEL = 'MINI (d=64  L=2 H=4 seq=64 )'

print(f'\nGPU    : {GPU_NAME}')
print(f'VRAM   : {VRAM_MB} MB | CC: sm_{GPU_CC}')
print(f'Config : {CONFIG_LABEL}')

# VRAM estimate
V_VOCAB   = 4096
param_b   = (V_VOCAB*D_MODEL*2 + N_LAYERS*(D_MODEL*D_MODEL*8+D_MODEL*4))*4
model_mb  = param_b/1024/1024
act_mb    = N_LAYERS*SEQ_LEN*D_MODEL*4*4/1024/1024
total_est = model_mb*3 + act_mb + 400
print(f'\nVRAM estimate: {total_est:.0f} MB / {VRAM_MB} MB ({total_est/VRAM_MB*100:.0f}%)')
print('✅ OK' if VRAM_MB-total_est > 500 else '⚠️  Tight')

os.system('nvcc --version | grep release')
os.system('free -h | head -2')

In [ ]:
# CELL 2 — 100MB Dataset
TARGET_MB    = 100
TARGET_BYTES = TARGET_MB * 1024 * 1024
print(f'Target: {TARGET_MB} MB dataset')

if os.path.exists(TRAIN_FILE) and os.path.getsize(TRAIN_FILE) >= int(TARGET_BYTES*0.8):
    sz = os.path.getsize(TRAIN_FILE)
    print(f'✅ Exists: {sz/1024/1024:.1f} MB')
else:
    done = False

    # 1. Local .txt (largest first)
    for fp in sorted(glob.glob('/kaggle/input/**/*.txt', recursive=True),
                     key=os.path.getsize, reverse=True):
        if os.path.getsize(fp) < TARGET_BYTES//4: continue
        print(f'Using local: {fp}')
        with open(fp,'r',errors='ignore') as fin, open(TRAIN_FILE,'w') as fout:
            written = 0
            for line in fin:
                fout.write(line); written += len(line)
                if written >= TARGET_BYTES: break
        done = True; break

    # 2. Local .jsonl
    if not done:
        for fp in sorted(glob.glob('/kaggle/input/**/*.jsonl', recursive=True),
                         key=os.path.getsize, reverse=True):
            if os.path.getsize(fp) < 1024*1024: continue
            print(f'Using JSONL: {fp}')
            written = 0
            with open(TRAIN_FILE,'w') as fout:
                with open(fp,'r',errors='ignore') as fin:
                    for line in fin:
                        if written >= TARGET_BYTES: break
                        try:
                            obj = json.loads(line)
                            t = obj.get('story') or obj.get('text') or ''
                            if t: fout.write(t.strip()+'\n\n'); written += len(t)
                        except: pass
            if written > TARGET_BYTES//4: done=True; break

    # 3. Download HuggingFace
    if not done:
        print('Downloading 150MB from HuggingFace...')
        TMP = '/kaggle/working/_tmp.txt'
        for url in [
            'https://huggingface.co/datasets/roneneldan/TinyStories/resolve/main/TinyStoriesV2-GPT4-train.txt',
            'https://huggingface.co/datasets/roneneldan/TinyStories/resolve/main/TinyStories-train.txt',
        ]:
            r = subprocess.run(['wget','-q','--timeout=180','--tries=3',url,'-O',TMP], capture_output=True)
            if r.returncode==0 and os.path.exists(TMP) and os.path.getsize(TMP)>1024*1024:
                with open(TMP,'r',errors='ignore') as fin, open(TRAIN_FILE,'w') as fout:
                    written=0
                    for line in fin:
                        fout.write(line); written+=len(line)
                        if written>=TARGET_BYTES: break
                if os.path.exists(TMP): os.remove(TMP)
                done=True; print('✅ Downloaded'); break

    # 4. Synthetic fallback
    if not done:
        print('No internet — high-quality synthetic data')
        stories = [
            'Once upon a time there was a little girl named Lily who loved to explore the forest near her home. Every morning she packed a small bag with bread and water. One day she found a hidden path that led to a clearing with a crystal blue pond. A family of ducks lived there and they were not afraid of her. She visited every day after school and gave them breadcrumbs. In return the ducks would waddle up to greet her. By winter Lily knew all their names.',
            'Tom was five years old and he had a best friend named Max who was a golden dog. They played together in the garden every afternoon after kindergarten. Max could catch a ball in midair and always brought it back. One rainy day they stayed inside and Tom read stories to Max. Max listened with his head tilted to one side. Toms mother took a photo and said it was the sweetest thing she had ever seen.',
            'In a village by the sea there lived a fisherman named Ben who had a small red boat. Every morning before sunrise he would row out to where the fish were plentiful. He always sang old songs to help him row. The other fishermen teased him but Ben did not mind. One season his catch was the best in the village. He said it was because the fish liked his singing.',
            'Sara and her little brother Jack built a cardboard castle in the living room. They used empty cereal boxes and toilet paper rolls for the towers. Their mother gave them old curtains to make a flag. The castle took all Saturday to build. They called it Castle Sunshine. When their father came home he pretended to be a dragon and they defended their castle with plastic swords. Everyone laughed until dinner time.',
            'The old tortoise named Slow lived in a garden where everyone knew him. He was very wise because he had seen many summers. The young rabbits often came to him with their problems. One spring a new family moved in and their puppy was very noisy. Slow simply moved under the bushes and waited. After a week the puppy grew curious and came to sniff him. Slow poked his head out and the puppy jumped back then wagged his tail. They became unlikely friends.',
        ]
        with open(TRAIN_FILE,'w') as f:
            w=0; i=0
            while w < TARGET_BYTES:
                s = stories[i % len(stories)]
                f.write(s+'\n\n'); w+=len(s)+2; i+=1

sz = os.path.getsize(TRAIN_FILE)
with open(TRAIN_FILE,'r',errors='ignore') as f:
    sample = f.read(400)
    f.seek(0)
    lines = sum(1 for _ in f)
print(f'\n✅ Dataset: {sz/1024/1024:.2f} MB | {lines:,} lines')
print('Sample:\n'+'-'*50+f'\n{sample[:350]}\n'+'-'*50)

In [ ]:
# CELL 3 — Clone / Update LOGOS
REPO_URL = 'https://github.com/Vikas8719/LOGOS.git'
if not os.path.exists(WORK_DIR):
    print('Cloning...')
    if os.system(f'git clone {REPO_URL} {WORK_DIR}') != 0:
        raise RuntimeError('Clone failed')
else:
    print('Pulling...')
    os.system(f'git -C {WORK_DIR} fetch origin main')
    os.system(f'git -C {WORK_DIR} reset --hard origin/main')

os.chdir(WORK_DIR)
print('Latest commits:')
os.system('git log --oneline -5')
required = ['cuda/train_gpu.cu','cuda/VedicGEMM.cu','cuda/ModelGPU.cu',
            'cuda/VedicGEMM.cuh','cuda/ModelGPU.cuh','src/main.cpp']
missing = [f for f in required if not os.path.exists(f'{WORK_DIR}/{f}')]
print('✅ All files present' if not missing else f'❌ Missing: {missing}')

In [ ]:
# CELL 4 — Patch Production Config into train_gpu.cu
GPU_SRC = f'{WORK_DIR}/cuda/train_gpu.cu'
with open(GPU_SRC) as f: src = f.read()

patches = [
    (r'cfg\.d_model\s*=\s*\d+;',     f'cfg.d_model     = {D_MODEL};'),
    (r'cfg\.num_layers\s*=\s*\d+;',   f'cfg.num_layers  = {N_LAYERS};'),
    (r'cfg\.num_heads\s*=\s*\d+;',    f'cfg.num_heads   = {N_HEADS};'),
    (r'cfg\.max_seq_len\s*=\s*\d+;',  f'cfg.max_seq_len = {SEQ_LEN};'),
    (r'cfg\.vocab_size\s*=\s*\d+;',   'cfg.vocab_size  = 4096;'),
    (r'float\s+lr_init\s*=\s*[\deE.+\-]+f;', 'float lr_init = 2e-4f;'),
    (r'int64_t\s+total_steps\s*=\s*\d+;', 'int64_t total_steps = 500000;'),
    (r'int\s+EPOCHS?\s*=\s*\d+;',    'int EPOCHS = 3;'),
]
for pat, rep in patches:
    new_src, n = re.subn(pat, rep, src)
    if n: src=new_src; print(f'  ✅ {rep.strip()[:55]}')
    else: print(f'  ⚠️  no match: {pat[:45]}')

with open(GPU_SRC,'w') as f: f.write(src)
print(f'\nConfig: d={D_MODEL} L={N_LAYERS} H={N_HEADS} seq={SEQ_LEN} | SHM v9 | FreeEnergy')

In [ ]:
# CELL 5 — Build GPU Binary
os.chdir(WORK_DIR)
os.system('rm -rf build && mkdir -p build')
print(f'Building sm_{GPU_CC} ({GPU_NAME})...')

build_start = time.time()
ret = os.system(f'''
cd {WORK_DIR} &&
cmake -B build -DCMAKE_BUILD_TYPE=Release \
  -DCMAKE_CUDA_ARCHITECTURES="{GPU_CC}" \
  -DCMAKE_CUDA_FLAGS="-O3 --use_fast_math" \
  -DCMAKE_CXX_FLAGS="-O3 -march=native -std=c++20" \
  2>&1 | tail -4 &&
cmake --build build --parallel $(nproc) 2>&1 | tee {BUILD_LOG}
''')
build_secs = time.time()-build_start

GPU_BIN = f'{WORK_DIR}/build/logos_gpu'
CPU_BIN = f'{WORK_DIR}/build/logos'
USING_GPU = os.path.exists(GPU_BIN)
BINARY = GPU_BIN if USING_GPU else CPU_BIN
MODE   = 'GPU' if USING_GPU else 'CPU'

print(f'Build: {build_secs:.1f}s | exit={ret} | using [{MODE}]')
if USING_GPU:
    print(f'✅ GPU binary: {os.path.getsize(GPU_BIN)/1024:.0f} KB')
else:
    print('⚠️  GPU build failed — check build_log.txt')
    if not os.path.exists(CPU_BIN):
        raise RuntimeError('Both builds failed!')

# Quick sanity
r = subprocess.run([BINARY,'--help'], capture_output=True, text=True, timeout=10)
print('✅ Binary OK' if r.returncode in [0,1] else f'⚠️  exit {r.returncode}')

In [ ]:
# CELL 6 — Pre-Training Speed Benchmark
print('=== Pre-Training Benchmark (forward pass only) ===')
t0 = time.time()
r  = subprocess.run([BINARY,'--benchmark'], capture_output=True,
                    text=True, timeout=120, cwd=WORK_DIR)
bench_secs = time.time()-t0
bench_out  = r.stdout + r.stderr
print(bench_out[:2500])

bench_stats = {}
for line in bench_out.split('\n'):
    m = re.search(r'([\d.]+)\s*tok(?:ens)?/s(?:ec)?', line, re.I)
    if m: bench_stats['fwd_tok_s'] = float(m.group(1))
    m = re.search(r'([\d.]+)\s*ms/(?:step|fwd)', line, re.I)
    if m: bench_stats['ms_per_fwd'] = float(m.group(1))

print(f'\nBenchmark: {bench_secs:.1f}s')
for k,v in bench_stats.items(): print(f'  {k}: {v}')

# Current VRAM after model load
r2 = subprocess.run(['nvidia-smi','--query-gpu=memory.used,memory.free',
                     '--format=csv,noheader,nounits'], capture_output=True, text=True)
p = r2.stdout.strip().split(',')
if len(p)>=2:
    print(f'VRAM after model: {p[0].strip()}MB used / {p[1].strip()}MB free')

In [ ]:
# CELL 7 — Monitor Setup
vram_log      = []
training_log  = []
step_times    = []
stop_mon      = threading.Event()

def monitor_vram():
    while not stop_mon.is_set():
        try:
            r = subprocess.run(
                ['nvidia-smi','--query-gpu=memory.used,memory.free,utilization.gpu',
                 '--format=csv,noheader,nounits'], capture_output=True, text=True)
            p = r.stdout.strip().split(',')
            if len(p)>=3:
                vram_log.append({'t':time.time(), 'used':int(p[0].strip()),
                                 'free':int(p[1].strip()), 'util':int(p[2].strip())})
                if int(p[1].strip()) < 400: print(f'⚠️  OOM RISK: {p[1].strip()}MB free!')
        except: pass
        time.sleep(15)

threading.Thread(target=monitor_vram, daemon=True).start()

# v9 SHM log format:
# Step | F | CE | Entropy | GNorm | α_H | α_L | T | Vedic
LOG_RE = re.compile(
    r'^\s*(\d+)\s*\|\s*([\d.]+)\s*\|\s*([\d.]+)\s*\|\s*([\d.]+)\s*\|'
    r'\s*([\d.]+)\s*\|\s*([\d.]+)\s*\|\s*([\d.]+)\s*\|\s*([0-9eE.+\-]+)\s*\|\s*(\S+)')

def parse(line):
    m = LOG_RE.match(line)
    if not m: return None
    try:
        return dict(step=int(m[1]),F=float(m[2]),CE=float(m[3]),S=float(m[4]),
                    gnorm=float(m[5]),aH=float(m[6]),aL=float(m[7]),
                    T=float(m[8]),vedic=m[9])
    except: return None

print('✅ Monitor + log parser ready')

In [ ]:
# CELL 8 — PRODUCTION TRAINING (100MB | 3 epochs)
os.chdir(WORK_DIR)
ds_mb = os.path.getsize(TRAIN_FILE)/1024/1024
training_log.clear(); step_times.clear(); vram_log.clear()
last_t = [time.time()]

print(f'╔══════════════════════════════════════════════╗')
print(f'║  LOGOS v3 — Production Training              ║')
print(f'║  {GPU_NAME[:46]:<46}║')
print(f'║  Data: {ds_mb:.0f}MB | Config: {CONFIG_LABEL[:27]:<27}║')
print(f'║  Optim: Hybrid SHM v9 (γ=0.1 β=0.9)        ║')
print(f'║  Loss:  Free Energy F = CE - T·S             ║')
print(f'╚══════════════════════════════════════════════╝\n')

train_start = time.time()
proc = subprocess.Popen([BINARY,'--train',TRAIN_FILE],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    universal_newlines=True, bufsize=1, cwd=WORK_DIR)

with open(LOG_FILE,'w') as logf:
    try:
        for raw in proc.stdout:
            line = raw.rstrip()
            print(line, flush=True)
            logf.write(raw); logf.flush()
            e = parse(line)
            if e:
                training_log.append(e)
                now = time.time()
                if len(training_log) > 1: step_times.append(now-last_t[0])
                last_t[0] = now
            if e and e['step'] % 1000==0 and vram_log:
                v=vram_log[-1]; el=(time.time()-train_start)/60
                tk=e['step']*SEQ_LEN; sp=tk/(time.time()-train_start)
                print(f'  ⟨{el:.0f}min | {sp:.0f} tok/s | VRAM {v["used"]}MB/{VRAM_MB}MB | util {v["util"]}%⟩')
    except KeyboardInterrupt:
        proc.terminate(); print('\n⏹  Stopped')

stop_mon.set()
ret  = proc.wait()
train_elapsed = time.time()-train_start

print(f'\n═══════════════════════════════════')
print(f'Time   : {train_elapsed/60:.1f} min | exit={ret}')
print(f'Steps  : {len(training_log)} logged entries')
if training_log:
    F0=training_log[0]['F']; Fe=training_log[-1]['F']
    FB=min(e['F'] for e in training_log)
    CE=training_log[-1]['CE']
    tk=training_log[-1]['step']*SEQ_LEN
    print(f'F      : {F0:.4f} → {Fe:.4f} (best {FB:.4f})')
    print(f'CE     : {CE:.4f} | PPL: {math.exp(min(CE,10)):.1f}')
    print(f'Tokens : {tk:,} @ {tk/train_elapsed:.0f} tok/s')
    d=F0-Fe
    print('✅✅ Excellent!' if d>3 else '✅ Good!' if d>1.5 else '⚠️  Need more steps' if d>0.5 else '❌ Check config')

In [ ]:
# CELL 9 — Full Dashboard (6 panels)
if not training_log and os.path.exists(LOG_FILE):
    with open(LOG_FILE) as f:
        for line in f:
            e=parse(line)
            if e: training_log.append(e)

if not training_log: print('No data'); exit()

def ema(v, a=0.03):
    s=v[0]; out=[]
    for x in v: s=(1-a)*s+a*x; out.append(s)
    return out

steps  = [e['step'] for e in training_log]
F_v    = [e['F']    for e in training_log]
CE_v   = [e['CE']   for e in training_log]
S_v    = [e['S']    for e in training_log]
aH_v   = [e['aH']   for e in training_log]
aL_v   = [e['aL']   for e in training_log]
T_v    = [e['T']    for e in training_log]
gn_v   = [e['gnorm'] for e in training_log]
Fs     = ema(F_v); CEs = ema(CE_v); Ss = ema(S_v)
PPL    = [math.exp(min(c,10)) for c in CEs]

# Speed
if step_times:
    interval = 100*SEQ_LEN
    sp_v  = [interval/dt for dt in step_times if dt>0]
    sp_st = steps[1:len(sp_v)+1]
else:
    tk=steps[-1]*SEQ_LEN; sp_avg=tk/train_elapsed
    sp_v=[sp_avg]*len(steps); sp_st=steps

peak_vram = max((v['used'] for v in vram_log), default=0)
vedic_e   = [e for e in training_log if e['vedic']!='N/A']
v_pass    = sum(1 for e in vedic_e if 'PASS' in e['vedic'].upper())

fig = plt.figure(figsize=(20,12))
fig.suptitle(f'LOGOS v3 | {GPU_NAME} | {CONFIG_LABEL} | {ds_mb:.0f}MB TinyStories',
             fontsize=12, fontweight='bold')
gs  = gridspec.GridSpec(2,3,figure=fig,hspace=0.4,wspace=0.35)

# 0: Free Energy
ax=fig.add_subplot(gs[0,0])
ax.plot(steps,F_v,'#3498db',lw=0.4,alpha=0.3,label='Raw')
ax.plot(steps,Fs,'#e74c3c',lw=2,label='EMA')
if F_v:
    ax.axhline(F_v[0],ls='--',lw=1,c='orange',label=f'Start {F_v[0]:.2f}')
    ax.axhline(min(F_v),ls='--',lw=1,c='green',label=f'Best {min(F_v):.2f}')
ax.set_title('Free Energy  F = CE - T·S'); ax.set_xlabel('Steps')
ax.legend(fontsize=8); ax.grid(alpha=0.3)

# 1: CE / S decomposition
ax=fig.add_subplot(gs[0,1])
ax.plot(steps,CEs,'#e74c3c',lw=1.5,label='CE')
ax.plot(steps,Ss,'#2ecc71',lw=1.5,label='Entropy S')
TS=[t*s for t,s in zip(ema(T_v),Ss)]
ax.plot(steps,TS,'#9b59b6',lw=1.2,ls='--',label='T·S bonus')
ax.set_title('CE vs Entropy Decomposition'); ax.set_xlabel('Steps')
ax.legend(fontsize=8); ax.grid(alpha=0.3)

# 2: Perplexity
ax=fig.add_subplot(gs[0,2])
ax.plot(steps,PPL,'#8e44ad',lw=2)
ax.fill_between(steps,PPL,alpha=0.12,color='#8e44ad')
ax.set_yscale('log'); ax.grid(alpha=0.3)
if PPL: ax.set_title(f'Perplexity  {PPL[0]:.0f} → {PPL[-1]:.0f}')
ax.set_xlabel('Steps'); ax.set_ylabel('PPL')

# 3: SHM annealing
ax=fig.add_subplot(gs[1,0])
ax.plot(steps,aH_v,'#e67e22',lw=2,label='α_H Hamiltonian')
ax.plot(steps,aL_v,'#3498db',lw=2,label='α_L Langevin')
axr=ax.twinx()
axr.plot(steps,T_v,'#95a5a6',lw=1,ls=':',label='T')
axr.set_yscale('log'); axr.set_ylabel('T',color='#95a5a6',fontsize=9)
ax.set_ylim(0,1.05); ax.axhline(0.5,c='gray',ls='--',lw=0.5)
ax.set_title('SHM Annealing  Explore → Exploit')
ax.set_xlabel('Steps'); ax.set_ylabel('Weight α')
ax.legend(fontsize=8,loc='center left'); ax.grid(alpha=0.3)

# 4: Speed
ax=fig.add_subplot(gs[1,1])
if sp_v:
    avg_sp=sum(sp_v)/len(sp_v)
    ax.plot(sp_st[:len(sp_v)],sp_v,'#1abc9c',lw=0.5,alpha=0.4)
    ax.plot(sp_st[:len(ema(sp_v,0.1))],ema(sp_v,0.1),'#16a085',lw=2)
    ax.axhline(avg_sp,c='red',ls='--',lw=1,label=f'Avg {avg_sp:.0f} tok/s')
    ax.set_title(f'Training Speed  avg {avg_sp:.0f} tok/s')
    ax.legend(fontsize=8)
ax.set_xlabel('Steps'); ax.set_ylabel('tok/s'); ax.grid(alpha=0.3)

# 5: VRAM
ax=fig.add_subplot(gs[1,2])
if vram_log:
    t0=vram_log[0]['t']
    vt=[(v['t']-t0)/60 for v in vram_log]
    vu=[v['used'] for v in vram_log]
    ut=[v.get('util',0) for v in vram_log]
    ax.fill_between(vt,vu,alpha=0.3,color='#e74c3c')
    ax.plot(vt,vu,'#e74c3c',lw=1.5,label=f'VRAM (peak {max(vu)}MB)')
    ax.axhline(VRAM_MB,c='red',ls='--',lw=1,label=f'Limit {VRAM_MB}MB')
    axr=ax.twinx(); axr.plot(vt,ut,'#3498db',lw=1,alpha=0.6)
    axr.set_ylabel('GPU util %',color='#3498db',fontsize=9)
    ax.set_title(f'VRAM  peak={max(vu)}MB ({max(vu)/VRAM_MB*100:.0f}%)')
    ax.legend(fontsize=8)
ax.set_xlabel('Time (min)'); ax.set_ylabel('MB'); ax.grid(alpha=0.3)

plt.savefig(f'{OUT_DIR}/dashboard_v3.png',dpi=150,bbox_inches='tight')
plt.show()
print(f'\n✅ Dashboard saved')
if vedic_e: print(f'Gunitasamuchayah: {v_pass}/{len(vedic_e)} PASS ({v_pass/len(vedic_e)*100:.0f}%)')

In [ ]:
# CELL 10 — Full Benchmark Report
if not training_log: print('No data'); exit()

tot_steps  = training_log[-1]['step']
tot_tokens = tot_steps * SEQ_LEN
tok_s      = tot_tokens/train_elapsed
ms_step    = train_elapsed/tot_steps*1000 if tot_steps else 0
F0=training_log[0]['F']; Fe=training_log[-1]['F']
FB=min(e['F'] for e in training_log)
CE_f=training_log[-1]['CE']; S_f=training_log[-1]['S']
PPL_f=math.exp(min(CE_f,10))
aH_f=training_log[-1]['aH']; T_f=training_log[-1]['T']
gn=[e['gnorm'] for e in training_log]
peak_v=max((v['used'] for v in vram_log),default=0)
avg_v =sum(v['used'] for v in vram_log)/len(vram_log) if vram_log else 0
avg_u =sum(v.get('util',0) for v in vram_log)/len(vram_log) if vram_log else 0

report = f"""
╔═══════════════════════════════════════════════════════════╗
║  LOGOS v3 — Production Benchmark Report                  ║
╠═══════════════════════════════════════════════════════════╣
║  HARDWARE                                                ║
║  GPU          : {GPU_NAME:<42}║
║  VRAM total   : {VRAM_MB} MB{' '*(40-len(str(VRAM_MB)))}║
║  VRAM peak    : {peak_v} MB  ({peak_v/VRAM_MB*100:.1f}%){' '*25}║
║  GPU util avg : {avg_u:.1f}%{' '*(41-len(f'{avg_u:.1f}'))}║
╠═══════════════════════════════════════════════════════════╣
║  TRAINING SPEED                                          ║
║  Tokens/sec   : {tok_s:,.0f}{' '*(42-len(f'{tok_s:,.0f}'))}║
║  ms/step      : {ms_step:.2f}{' '*(42-len(f'{ms_step:.2f}'))}║
║  M tok/min    : {tok_s*60/1e6:.3f}{' '*(42-len(f'{tok_s*60/1e6:.3f}'))}║
║  Total steps  : {tot_steps:,}{' '*(42-len(f'{tot_steps:,}'))}║
║  Total tokens : {tot_tokens:,}{' '*(42-len(f'{tot_tokens:,}'))}║
║  Wall time    : {train_elapsed/3600:.2f} hr ({train_elapsed/60:.0f} min){' '*20}║
╠═══════════════════════════════════════════════════════════╣
║  MODEL: {CONFIG_LABEL:<50}║
╠═══════════════════════════════════════════════════════════╣
║  CONVERGENCE (Free Energy F = CE - T·S)                  ║
║  F  : {F0:.4f} → {Fe:.4f}  (best {FB:.4f}  drop {F0-Fe:.4f}){' '*6}║
║  CE : {CE_f:.4f}  |  S: {S_f:.4f}  |  PPL: {PPL_f:.1f}{' '*(25-len(f'{PPL_f:.1f}'))}║
╠═══════════════════════════════════════════════════════════╣
║  HYBRID SHM v9 (γ=0.10 β=0.90)                          ║
║  Final α_H    : {aH_f:.3f}  (Hamiltonian){' '*24}║
║  Final α_L    : {1-aH_f:.3f}  (Langevin){' '*27}║
║  Final T      : {T_f:.2e}{' '*(41-len(f'{T_f:.2e}'))}║
╠═══════════════════════════════════════════════════════════╣
║  GRADIENT HEALTH                                         ║
║  avg  ‖∇‖    : {sum(gn)/len(gn):.4f}{' '*(42-len(f'{sum(gn)/len(gn):.4f}'))}║
║  max  ‖∇‖    : {max(gn):.4f}{' '*(42-len(f'{max(gn):.4f}'))}║
║  clip events  : {sum(1 for g in gn if g>=0.95)}{' '*(42-len(str(sum(1 for g in gn if g>=0.95))))}║
╠═══════════════════════════════════════════════════════════╣
║  VEDIC (Gunitasamuchayah)  {v_pass}/{len(vedic_e)} PASS ({v_pass/max(len(vedic_e),1)*100:.0f}%){' '*14}║
╚═══════════════════════════════════════════════════════════╝
"""
print(report)
with open(f'{OUT_DIR}/benchmark_report.txt','w') as f: f.write(report)
print('✅ Report saved')

In [ ]:
# CELL 11 — Feynman Beam Search Generation
os.chdir(WORK_DIR)
ckpts = sorted([f for f in glob.glob(f'{WORK_DIR}/*.bin')
                if 'vocab' not in os.path.basename(f).lower()])
print(f'Checkpoints: {len(ckpts)}')
for c in ckpts[-3:]: print(f'  {os.path.basename(c)}')

if not ckpts:
    print('No checkpoint found yet'); exit()

ckpt = ckpts[-1]
print(f'\nUsing: {os.path.basename(ckpt)}')

prompts = ['Once upon a time','The little girl named Lily',
           'Tom had a red ball','In a small village']

print('\n'+'═'*60)
print('FEYNMAN BEAM SEARCH  ħ=1.0  beams=4  top_k=50')
print('═'*60)
for p in prompts:
    print(f"\n▶ '{p}'")
    r = subprocess.run([BINARY,'--generate',ckpt,p,'64','4','1.0','50'],
        capture_output=True, text=True, timeout=120, cwd=WORK_DIR)
    for line in (r.stdout+r.stderr).split('\n'):
        if any(k in line for k in ['Generated:','Beam ','Action ']): print(line)
    if not any(k in (r.stdout+r.stderr) for k in ['Generated:','Beam']):
        print((r.stdout+r.stderr)[:400])

print('\n'+'═'*60)
print('ħ COMPARISON: Classical vs Standard vs Quantum')
print('═'*60)
tp = 'Once upon a time'
for hbar,lbl in [('0.3','Classical  ħ=0.3 (sharp)'),
                 ('1.0','Standard   ħ=1.0 (beam)'),
                 ('1.5','Quantum    ħ=1.5 (diverse)')]:
    print(f'\n  [{lbl}]')
    r = subprocess.run([BINARY,'--generate',ckpt,tp,'48','2',hbar,'30'],
        capture_output=True, text=True, timeout=60, cwd=WORK_DIR)
    for line in (r.stdout+r.stderr).split('\n'):
        if 'Generated:' in line: print(f'  {line.strip()}')

In [ ]:
# CELL 12 — Physics Impact Plot
fig, axes = plt.subplots(2,2,figsize=(16,10))
fig.suptitle('LOGOS v3 — Physics Component Impact', fontsize=13, fontweight='bold')

# α_H vs Loss scatter
ax=axes[0,0]
sc=ax.scatter([e['aH'] for e in training_log],[e['F'] for e in training_log],
              c=range(len(training_log)),cmap='viridis',s=6,alpha=0.5)
plt.colorbar(sc,ax=ax,label='Progress →')
ax.set_xlabel('α_H (Hamiltonian weight)'); ax.set_ylabel('F')
ax.set_title('SHM Annealing: α_H vs F\n(dark=early, bright=late)')
ax.grid(alpha=0.3)

# Free Energy stack
ax=axes[0,1]
CEs=ema(CE_v); Ss=ema(S_v); Ts=ema(T_v)
TSv=[t*s for t,s in zip(Ts,Ss)]
CE_net=[max(c-ts,0) for c,ts in zip(CEs,TSv)]
ax.stackplot(steps,CE_net,TSv,
    labels=['CE - T·S (net)','T·S (entropy bonus)'],
    colors=['#e74c3c','#2ecc71'],alpha=0.7)
ax.plot(steps,Fs,'k',lw=2,label='F = CE - T·S')
ax.set_title('Free Energy Decomposition'); ax.set_xlabel('Steps')
ax.legend(fontsize=8); ax.grid(alpha=0.3)

# Nikhilam INT8 savings
ax=axes[1,0]
DH=D_MODEL//N_HEADS
seqs=list(range(32,513,32))
f32=[2*s*DH*N_HEADS*N_LAYERS*4/1024/1024 for s in seqs]
i8 =[2*s*DH*N_HEADS*N_LAYERS*1/1024/1024 for s in seqs]
ax.fill_between(seqs,f32,i8,alpha=0.4,color='#e74c3c',label='Saved by INT8')
ax.plot(seqs,f32,'r-',lw=2,label='Float32 KV')
ax.plot(seqs,i8,'g-',lw=2,label='Nikhilam INT8')
ax.axvline(SEQ_LEN,c='blue',ls='--',lw=1.5,label=f'seq={SEQ_LEN}')
ax.set_title(f'Nikhilam INT8 VRAM Savings\nd={D_MODEL} L={N_LAYERS} → 4x')
ax.set_xlabel('Seq len'); ax.set_ylabel('MB'); ax.legend(fontsize=8); ax.grid(alpha=0.3)

# Gradient norm
ax=axes[1,1]
ax.plot(steps,gn_v,'#3498db',lw=0.4,alpha=0.3)
ax.plot(steps,ema(gn_v,0.05),'#2980b9',lw=2)
ax.axhline(1.0,c='red',ls='--',lw=1,label='Clip=1.0')
ax.axhline(0.1,c='orange',ls=':',lw=1)
ax.fill_between(steps,0.05,0.8,alpha=0.05,color='green')
ax.set_title(f'Grad Norm | avg={sum(gn_v)/len(gn_v):.3f}')
ax.legend(fontsize=8); ax.grid(alpha=0.3)
ax.set_ylim(0,min(max(gn_v)*1.1,2.0))

plt.tight_layout()
plt.savefig(f'{OUT_DIR}/physics_v3.png',dpi=150,bbox_inches='tight')
plt.show()
print('✅ Physics plot saved')

In [ ]:
# CELL 13 — Save Everything
print('Saving outputs...')
saved=[]

def cp(src,label=''):
    if os.path.exists(src):
        dst=os.path.join(OUT_DIR,os.path.basename(src))
        shutil.copy2(src,dst)
        print(f'  ✅ {label or os.path.basename(src)}: {os.path.getsize(dst)/1024:.1f} KB')
        saved.append(dst); return True
    return False

ckpts=sorted([f for f in glob.glob(f'{WORK_DIR}/*.bin') if 'vocab' not in f.lower()])
if ckpts: cp(ckpts[-1],'Best checkpoint')
cp(f'{WORK_DIR}/vocab.bin','Vocab')
cp(LOG_FILE,'Training log'); cp(BUILD_LOG,'Build log')

if training_log:
    summary={
        'version':'v3','gpu':GPU_NAME,'vram_mb':VRAM_MB,
        'config':{'d':D_MODEL,'L':N_LAYERS,'H':N_HEADS,'seq':SEQ_LEN},
        'dataset_mb':round(ds_mb,2),
        'speed':{'tok_s':round(tok_s),'ms_step':round(ms_step,2),
                 'tot_steps':tot_steps,'tot_tokens':tot_tokens,
                 'wall_hr':round(train_elapsed/3600,3)},
        'loss':{'F0':round(F0,4),'Fe':round(Fe,4),'Fbest':round(FB,4),
                'drop':round(F0-Fe,4),'CE':round(CE_f,4),'PPL':round(PPL_f,1)},
        'physics':{'aH_final':round(aH_f,3),'T_final':float(f'{T_f:.2e}'),
                   'vedic_pass':v_pass,'vedic_total':len(vedic_e)},
        'vram':{'peak_mb':peak_v,'avg_mb':round(avg_v),'util_pct':round(avg_u,1)},
    }
    spath=f'{OUT_DIR}/training_summary.json'
    with open(spath,'w') as f: json.dump(summary,f,indent=2)
    print(f'  ✅ training_summary.json'); saved.append(spath)

print(f'\n✅ {len(saved)} files in {OUT_DIR}')
os.system(f'ls -lh {OUT_DIR}')

print()
print('╔══════════════════════════════════════════════╗')
print('║  LOGOS v3 Production Complete! ✅            ║')
if training_log:
    print(f'║  F: {F0:.3f} → {Fe:.3f}  PPL: {PPL_f:.0f}  tok/s: {tok_s:,.0f}{" "*3}║')
    print(f'║  VRAM: {peak_v}MB / {VRAM_MB}MB  GPU: {avg_u:.0f}% util{" "*8}║')
print('║  Download from Kaggle Output tab! 📥        ║')
print('╚══════════════════════════════════════════════╝')